In [1]:
# Install dependencies
!pip install --quiet kagglehub scikit-learn surprise pandas matplotlib seaborn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 11.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score, recall_score, f1_score


from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score



In [3]:
# Download dataset from Kaggle
path = kagglehub.dataset_download("arashnic/book-recommendation-dataset")
print("Dataset path:", path)

# List all CSV files in the folder
csvs = glob.glob(os.path.join(path, "*.csv"))
print("CSV files found:", csvs)

# Load data
books = pd.read_csv(os.path.join(path, "Books.csv"), encoding="latin-1")
users = pd.read_csv(os.path.join(path, "Users.csv"), encoding="latin-1")
ratings = pd.read_csv(os.path.join(path, "Ratings.csv"), encoding="latin-1")

print("Books:", books.shape)
print("Users:", users.shape)
print("Ratings:", ratings.shape)


100%|██████████| 24.3M/24.3M [00:02<00:00, 11.4MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/arashnic/book-recommendation-dataset/versions/3
CSV files found: ['/root/.cache/kagglehub/datasets/arashnic/book-recommendation-dataset/versions/3/Users.csv', '/root/.cache/kagglehub/datasets/arashnic/book-recommendation-dataset/versions/3/Ratings.csv', '/root/.cache/kagglehub/datasets/arashnic/book-recommendation-dataset/versions/3/Books.csv']


/tmp/ipython-input-1429076803.py:10: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv(os.path.join(path, "Books.csv"), encoding="latin-1")


Books: (271360, 8)
Users: (278858, 3)
Ratings: (1149780, 3)


In [4]:
# Download dataset
path = kagglehub.dataset_download("arashnic/book-recommendation-dataset")
print("Dataset path:", path)

# Check CSV files
csvs = glob.glob(os.path.join(path, "*.csv"))
print("Found CSVs:", csvs)

# Typically: Books.csv, Users.csv, Ratings.csv
books = pd.read_csv(os.path.join(path, "Books.csv"), encoding="latin-1")
users = pd.read_csv(os.path.join(path, "Users.csv"), encoding="latin-1")
ratings = pd.read_csv(os.path.join(path, "Ratings.csv"), encoding="latin-1")

print("Books:", books.shape, "Users:", users.shape, "Ratings:", ratings.shape)


Using Colab cache for faster access to the 'book-recommendation-dataset' dataset.
Dataset path: /kaggle/input/book-recommendation-dataset
Found CSVs: ['/kaggle/input/book-recommendation-dataset/Ratings.csv', '/kaggle/input/book-recommendation-dataset/Users.csv', '/kaggle/input/book-recommendation-dataset/Books.csv']


/tmp/ipython-input-3202357191.py:10: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv(os.path.join(path, "Books.csv"), encoding="latin-1")


Books: (271360, 8) Users: (278858, 3) Ratings: (1149780, 3)


In [5]:
# Keep useful columns only
books = books[["ISBN", "Book-Title", "Book-Author", "Year-Of-Publication", "Publisher"]]
ratings = ratings[["User-ID", "ISBN", "Book-Rating"]]

# Filter out zero ratings (implicit feedback problem)
ratings = ratings[ratings["Book-Rating"] > 0]

print("Unique users:", ratings["User-ID"].nunique())
print("Unique books:", ratings["ISBN"].nunique())
ratings.head()


Unique users: 77805
Unique books: 185973


,User-ID,ISBN,Book-Rating
1,276726,0155061224,5
3,276729,052165615X,3
4,276729,0521795028,6
6,276736,3257224281,8
7,276737,0600570967,6


In [ ]:
# Use TF-IDF on book titles + authors + publishers
books["combined"] = (
    books["Book-Title"].astype(str) + " " +
    books["Book-Author"].astype(str) + " " +
    books["Publisher"].astype(str)
)

# Vectorize text
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(books["combined"])

# Compute cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Function to recommend similar books
book_index = pd.Series(books.index, index=books["Book-Title"].str.lower())

def recommend_books_content(title, n=5):
    idx = book_index.get(title.lower())
    if idx is None:
        return ["Book not found in dataset."]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:n+1]
    indices = [i[0] for i in sim_scores]
    return books.iloc[indices]["Book-Title"].tolist()

print("Recommendations for 'Harry Potter and the Chamber of Secrets':")
print(recommend_books_content("Harry Potter and the Chamber of Secrets", n=5))


In [ ]:
# Prepare ratings data for Surprise
reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(ratings[["User-ID", "ISBN", "Book-Rating"]], reader)

# Train-test split
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Train SVD model
svd = SVD()
svd.fit(trainset)

# Evaluate on test
predictions = svd.test(testset)
print("RMSE:", accuracy.rmse(predictions))
def recommend_books_hybrid(user_id, n=5):
    # If user exists → collaborative filtering
    if user_id in ratings["User-ID"].values:
        user_ratings = ratings[ratings["User-ID"] == user_id].merge(books, on="ISBN")
        rated_books = user_ratings["Book-Title"].tolist()

        # Predict ratings for unseen books
        all_books = books["ISBN"].tolist()
        preds = [svd.predict(user_id, isbn) for isbn in all_books]
        preds = sorted(preds, key=lambda x: x.est, reverse=True)[:n]
        return books.set_index("ISBN").loc[[p.iid for p in preds]]["Book-Title"].tolist()

    # Cold-start: recommend popular books
    else:
        popular = ratings.groupby("ISBN").size().sort_values(ascending=False).head(n).index
        return books.set_index("ISBN").loc[popular]["Book-Title"].tolist()

print("Hybrid recommendations for user 276729:")
print(recommend_books_hybrid(276729, n=5))

print("Hybrid recommendations for NEW user (cold-start):")
print(recommend_books_hybrid(-1, n=5))


In [ ]:
# Convert Surprise predictions into binary relevance (liked if rating >= 7)
y_true = [1 if true_r >= 7 else 0 for (_, _, true_r, _) in testset]
y_pred = [1 if pred.est >= 7 else 0 for pred in predictions]

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)
